# 9. Autoencoders

En este notebook aprenderás los fundamentos de los autoencoders, incluyendo autoencoders simples, denoising y variacionales (VAE), con ejemplos prácticos en TensorFlow/Keras.

## Objetivo
- Comprender la teoría y arquitectura de los autoencoders.
- Implementar un autoencoder simple, un denoising autoencoder y un VAE.
- Visualizar reconstrucciones y el espacio latente.
- Aplicar buenas prácticas para entrenamiento eficiente.

## Prerequisitos

> 📌 **Prerequisitos:** Haber completado los notebooks [04 (MLP)](./04_redes_neuronales_capa_densa.ipynb) y [08 (GANs)](./08_gans.ipynb).

- Conceptos de redes neuronales y modelos generativos.

## 1. Introducción teórica

Un autoencoder comprime datos en una representación de menor dimensión y luego los reconstruye.

| Tipo | Uso principal | Característica |
|------|---------------|----------------|
| **Simple** | Reducción de dimensionalidad | Comprime y reconstruye |
| **Denoising** | Eliminación de ruido | Entrena con datos ruidosos |
| **Variacional (VAE)** | Generación de datos | Espacio latente continuo y muestreable |

## 2. Importación de librerías

In [ ]:
# === Reproducibilidad ===
import random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
tf.random.set_seed(SEED)

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 3. Autoencoder simple

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0
X_train_flat = X_train.reshape(-1, 28*28)
X_test_flat = X_test.reshape(-1, 28*28)

encoding_dim = 32
input_img = keras.Input(shape=(784,))
encoded = keras.layers.Dense(128, activation='relu')(input_img)
encoded = keras.layers.Dense(encoding_dim, activation='relu')(encoded)
decoded = keras.layers.Dense(128, activation='relu')(encoded)
decoded = keras.layers.Dense(784, activation='sigmoid')(decoded)

autoencoder = keras.Model(input_img, decoded)
autoencoder.compile(optimizer='adam', loss='mse')

autoencoder.fit(X_train_flat, X_train_flat, epochs=20, batch_size=256,
                shuffle=True, validation_data=(X_test_flat, X_test_flat), verbose=1)

### Visualización de reconstrucciones

In [ ]:
decoded_imgs = autoencoder.predict(X_test_flat)

n = 10
plt.figure(figsize=(20, 4))
for i in range(n):
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(X_test[i], cmap='gray')
    plt.title('Original')
    plt.axis('off')
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(decoded_imgs[i].reshape(28, 28), cmap='gray')
    plt.title('Reconstruida')
    plt.axis('off')
plt.suptitle('Autoencoder Simple')
plt.show()

## 4. Denoising Autoencoder

In [ ]:
noise_factor = 0.5
X_train_noisy = X_train_flat + noise_factor * np.random.normal(0, 1, X_train_flat.shape)
X_test_noisy = X_test_flat + noise_factor * np.random.normal(0, 1, X_test_flat.shape)
X_train_noisy = np.clip(X_train_noisy, 0., 1.)
X_test_noisy = np.clip(X_test_noisy, 0., 1.)

denoising_ae = keras.models.clone_model(autoencoder)
denoising_ae.compile(optimizer='adam', loss='mse')
denoising_ae.fit(X_train_noisy, X_train_flat, epochs=10, batch_size=256,
                 shuffle=True, validation_data=(X_test_noisy, X_test_flat), verbose=1)

decoded_noisy = denoising_ae.predict(X_test_noisy)

plt.figure(figsize=(20, 6))
for i in range(n):
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(X_test_noisy[i].reshape(28, 28), cmap='gray')
    plt.title('Ruidosa')
    plt.axis('off')
    ax = plt.subplot(3, n, i + 1 + n)
    plt.imshow(decoded_noisy[i].reshape(28, 28), cmap='gray')
    plt.title('Limpiada')
    plt.axis('off')
    ax = plt.subplot(3, n, i + 1 + 2*n)
    plt.imshow(X_test[i], cmap='gray')
    plt.title('Original')
    plt.axis('off')
plt.suptitle('Denoising Autoencoder')
plt.show()

## 5. Variational Autoencoder (VAE)

El VAE aprende un espacio latente **continuo y muestreable**, permitiendo generar nuevos datos interpolando en el espacio latente.

In [ ]:
# VAE simplificado
latent_dim = 2  # 2D para poder visualizar

# Encoder
encoder_inputs = keras.Input(shape=(784,))
h = keras.layers.Dense(256, activation='relu')(encoder_inputs)
z_mean = keras.layers.Dense(latent_dim)(h)
z_log_var = keras.layers.Dense(latent_dim)(h)

# Reparametrización
def sampling(args):
    z_mean, z_log_var = args
    epsilon = tf.random.normal(shape=tf.shape(z_mean))
    return z_mean + tf.exp(0.5 * z_log_var) * epsilon

z = keras.layers.Lambda(sampling)([z_mean, z_log_var])

# Decoder
decoder_h = keras.layers.Dense(256, activation='relu')
decoder_out = keras.layers.Dense(784, activation='sigmoid')
h_decoded = decoder_h(z)
x_decoded = decoder_out(h_decoded)

vae = keras.Model(encoder_inputs, x_decoded)

# Loss: reconstrucción + KL divergence
reconstruction_loss = keras.losses.binary_crossentropy(encoder_inputs, x_decoded) * 784
kl_loss = -0.5 * tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=-1)
vae_loss = tf.reduce_mean(reconstruction_loss + kl_loss)
vae.add_loss(vae_loss)
vae.compile(optimizer='adam')

vae.fit(X_train_flat, None, epochs=15, batch_size=256, validation_data=(X_test_flat, None), verbose=1)

In [ ]:
# Visualizar espacio latente
encoder = keras.Model(encoder_inputs, z_mean)
z_test = encoder.predict(X_test_flat)

plt.figure(figsize=(8, 6))
plt.scatter(z_test[:, 0], z_test[:, 1], c=y_test, cmap='tab10', s=2, alpha=0.7)
plt.colorbar(label='Dígito')
plt.title('Espacio latente del VAE (2D)')
plt.xlabel('z[0]')
plt.ylabel('z[1]')
plt.show()

### Recomendaciones prácticas

| Aspecto | Recomendación |
|---------|---------------|
| **Espacio latente** | Ajusta dimensión según la compresión deseada |
| **Regularización** | Usa Dropout, L1/L2 para evitar sobreajuste |
| **Denoising** | Mejora la robustez y las representaciones |
| **VAE** | Espacio latente continuo; ideal para generación |
| **Visualización** | Siempre visualiza reconstrucciones |

## 6. Discusión y Conclusiones

- Implementamos 3 tipos de autoencoders: simple, denoising y variacional.
- El VAE permite generar nuevos datos y explorar el espacio latente.
- Los denoising autoencoders son útiles para limpiar datos.
- En el siguiente notebook veremos técnicas de clustering y reducción de dimensionalidad.

## 7. Ejercicios Propuestos

1. **Ejercicio 1:** Cambia `encoding_dim` a 8, 16 y 64. ¿Cómo afecta la calidad de reconstrucción?

2. **Ejercicio 2:** Usa el VAE para generar nuevos dígitos muestreando puntos del espacio latente.

3. **Ejercicio 3:** Implementa un autoencoder convolucional (usando Conv2D y Conv2DTranspose).

4. **Ejercicio 4 (Avanzado):** Usa un VAE condicional (CVAE) que genere dígitos específicos.

## 8. Referencias y Recursos

- [TensorFlow Autoencoder Tutorial](https://www.tensorflow.org/tutorials/generative/autoencoder)
- [VAE Tutorial - Keras](https://keras.io/examples/generative/vae/)
- Kingma & Welling (2014). *Auto-Encoding Variational Bayes.*

---

📎 **Notebook anterior:** [08. Redes Generativas (GANs)](./08_gans.ipynb)  
📎 **Notebook siguiente:** [10. Clustering y Reducción de Dimensionalidad](./10_clustering_reduccion_dimensionalidad.ipynb)